In [1]:
#run this to create local directory for credentials for Snowflake

In [2]:
import boto3
import json

def store_secret(secret_name, username, password, region_name="us-east-1"):
    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )
    
    # Create the secret value
    secret_value = json.dumps({
        'username': username,
        'password': password
    })
    
    try:
        # Create a new secret
        response = client.create_secret(
            Name=secret_name,
            SecretString=secret_value
        )
        return response
    except client.exceptions.ResourceExistsException:
        # Update existing secret
        response = client.update_secret(
            SecretId=secret_name,
            SecretString=secret_value
        )
        return response
    
def get_secret(secret_name, region_name="us-east-1"):
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )
    
    try:
        response = client.get_secret_value(SecretId=secret_name)
        secret = json.loads(response['SecretString'])
        return secret
    except Exception as e:
        print(f"Error retrieving secret: {str(e)}")
        return None
    
import os

def export_credentials_to_files(secret_name, credentials_dir="credentials", region_name="us-east-1"):
    # Get secrets from AWS
    credentials = get_secret(secret_name, region_name)
    if not credentials:
        print("Failed to retrieve credentials")
        return
    
    # Create credentials directory if it doesn't exist
    os.makedirs(credentials_dir, exist_ok=True)
    
    # Write username to file
    with open(f"{credentials_dir}/mlAccountUsername", "w") as f:
        f.write(credentials['username'])
    
    # Write password to file
    with open(f"{credentials_dir}/mlAccountCode", "w") as f:
        f.write(credentials['password'])
    
    print(f"Credentials exported to {credentials_dir}/")    

In [3]:
import boto3

session = boto3.session.Session()
client = session.client('secretsmanager')

try:
    response = client.list_secrets()
    print("Successfully listed secrets!")
    print(response)
except Exception as e:
    print(f"Error: {str(e)}")

Successfully listed secrets!
{'SecretList': [{'ARN': 'arn:aws:secretsmanager:us-east-1:[REDACTED]:secret:AmazonSageMaker-aws[REDACTED]AppliedScience-p00TwH', 'Name': 'AmazonSageMaker-aws[REDACTED]AppliedScience', 'LastChangedDate': datetime.datetime(2022, 8, 30, 20, 10, 7, 860000, tzinfo=tzlocal()), 'LastAccessedDate': datetime.datetime(2022, 8, 30, 0, 0, tzinfo=tzlocal()), 'SecretVersionsToStages': {'0c86c676-79b6-44c1-bf9c-ad1a0f7c95d7': ['AWSCURRENT']}, 'CreatedDate': datetime.datetime(2022, 8, 30, 20, 10, 7, 561000, tzinfo=tzlocal())}, {'ARN': 'arn:aws:secretsmanager:us-east-1:[REDACTED]:secret:AmazonSageMaker-omarAppliedScienceGithub-puS7z7', 'Name': 'AmazonSageMaker-omarAppliedScienceGithub', 'LastChangedDate': datetime.datetime(2022, 8, 30, 20, 42, 44, 357000, tzinfo=tzlocal()), 'LastAccessedDate': datetime.datetime(2022, 8, 30, 0, 0, tzinfo=tzlocal()), 'SecretVersionsToStages': {'8f22da0a-2847-4935-b624-6fc98833e004': ['AWSCURRENT']}, 'CreatedDate': datetime.datetime(2022, 8, 3

In [4]:
# To store
#uncomment and fill in your crednetials here
#store_secret('omar-ml-credentials', username=, password=)


{'ARN': 'arn:aws:secretsmanager:us-east-1:[REDACTED]:secret:omar-ml-credentials-ylwLId',
 'Name': 'omar-ml-credentials',
 'VersionId': '3a13b726-722e-43c0-a2b6-526468f721cc',
 'ResponseMetadata': {'RequestId': 'e33fa091-5c23-4c8c-963b-4a354938ec30',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'e33fa091-5c23-4c8c-963b-4a354938ec30',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '169',
   'date': 'Sat, 04 Jan 2025 22:22:47 GMT'},
  'RetryAttempts': 0}}

In [5]:
# To retrieve
credentials = get_secret('omar-ml-credentials')
if credentials:
    username = credentials['username']
    password = credentials['password']

In [7]:
# This will create credentials/mlAccountUsername and credentials/mlAccountCode
export_credentials_to_files('omar-ml-credentials')

Credentials exported to credentials/
